In [0]:
# =====================================================================================
# Notebook: 01_bronze/02_files_to_bronze_delta_v2.py
# Project : nba-datalakehouse
# Purpose : Convert raw JSONL landing files (Unity Catalog Volume) into Bronze Delta tables
#           with v2 semantics:
#             - Append new dt partitions without wiping history
#             - Idempotent per-dt reloads (overwrite ONLY the dt partitions you choose)
#             - Robust file reading (recursive) so nested folders are included
#
# KEY IDEA (V2)
# -------------
# 1) Read landing files from the dataset root with recursiveFileLookup
# 2) Determine which dt values exist in landing (or for a chosen date range)
# 3) For each dt, write only that dt partition into the Bronze table
#    using replaceWhere="dt = 'YYYY-MM-DD'"
#
# This gives you:
#   - Daily runs: overwrite just today’s dt partition (safe reruns)
#   - Catch-up runs: overwrite a range of dt partitions (safe reruns)
#   - No accidental full-table overwrite
#
# IMPORTANT NOTE
# --------------
# Delta + Unity Catalog can be strict about schema changes. In v2 we:
#   - Create table if missing
#   - For existing tables, we keep schema stable by selecting consistent columns
#     (for now we write the full inferred schema and enforce dt as string)
# =====================================================================================

from pyspark.sql import functions as F

LANDING_BASE = "dbfs:/Volumes/workspace/nba_files/landing"

DATASETS = [
    ("balldontlie",  "teams",  "workspace.nba_bronze.balldontlie_teams"),
    ("balldontlie",  "games",  "workspace.nba_bronze.balldontlie_games"),
    ("the_odds_api", "sports", "workspace.nba_bronze.the_odds_api_sports"),
    ("the_odds_api", "events", "workspace.nba_bronze.the_odds_api_events"),
    ("the_odds_api", "odds",   "workspace.nba_bronze.the_odds_api_odds"),
]

MAX_DT_TO_LOAD = 30

def _exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def landing_entity_root(source: str, entity: str) -> str:
    return f"{LANDING_BASE}/{source}/{entity}"

def list_dt_partitions(root: str):
    dts = []
    for x in dbutils.fs.ls(root):
        name = x.name.rstrip("/")
        if name.startswith("dt="):
            dts.append(name.split("dt=")[1])
    return sorted(dts)

def read_dataset_root(root: str):
    return (
        spark.read
        .option("recursiveFileLookup", "true")
        .json(root)
    )

def add_dt(df):
    return df.withColumn("dt", F.col("meta.dt").cast("string"))

def normalize_bronze(df):
    return (
        df.select(
            F.col("dt"),
            F.to_json(F.col("meta")).alias("meta"),
            F.to_json(F.col("data")).alias("data"),
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_loaded_at"),
        )
    )

def table_exists(table_fqn: str) -> bool:
    try:
        spark.table(table_fqn)
        return True
    except Exception:
        return False

def ensure_table_created(df_norm, table_fqn: str):
    if table_exists(table_fqn):
        return
    (df_norm.limit(0)
        .write
        .format("delta")
        .mode("overwrite")
        .partitionBy("dt")
        .saveAsTable(table_fqn))

results = []

for source, entity, table_fqn in DATASETS:
    root = landing_entity_root(source, entity)

    if not _exists(root):
        print(f"SKIP: missing landing path for {source}/{entity}: {root}")
        continue

    all_dts = list_dt_partitions(root)
    if not all_dts:
        print(f"SKIP: no dt= partitions found for {source}/{entity}: {root}")
        continue

    load_dts = all_dts[-MAX_DT_TO_LOAD:] if MAX_DT_TO_LOAD else all_dts

    print("\n=== DATASET ===")
    print("source/entity:", source, entity)
    print("landing root:", root)
    print("table:", table_fqn)
    print("dt partitions found:", len(all_dts), "| loading:", len(load_dts))
    print("last 5 dt:", all_dts[-5:])

    df_all = read_dataset_root(root)

    if "meta" not in df_all.columns:
        print(f"SKIP: {source}/{entity} missing 'meta' column")
        continue

    df_all = add_dt(df_all)

    if "dt" not in df_all.columns:
        print(f"SKIP: {source}/{entity} could not derive dt from meta.dt")
        continue

    df_norm = normalize_bronze(df_all)

    ensure_table_created(df_norm, table_fqn)

    loaded_dts = []
    total_rows = 0

    for dt in load_dts:
        df_dt = df_norm.filter(F.col("dt") == F.lit(dt))
        n = df_dt.count()
        if n == 0:
            continue

        (df_dt.write
             .format("delta")
             .mode("overwrite")
             .option("replaceWhere", f"dt = '{dt}'")
             .saveAsTable(table_fqn))

        loaded_dts.append(dt)
        total_rows += n
        print(f"Loaded dt={dt} rows={n}")

    results.append((table_fqn, root, len(loaded_dts), total_rows, loaded_dts[-3:] if loaded_dts else []))
    print(f"Done {table_fqn}: loaded_dt_count={len(loaded_dts)} total_rows={total_rows}")

display(results)


In [0]:
%sql

DROP TABLE IF EXISTS workspace.nba_bronze.balldontlie_games;
DROP TABLE IF EXISTS workspace.nba_bronze.balldontlie_teams;
DROP TABLE IF EXISTS workspace.nba_bronze.the_odds_api_sports;
DROP TABLE IF EXISTS workspace.nba_bronze.the_odds_api_events;
DROP TABLE IF EXISTS workspace.nba_bronze.the_odds_api_odds;
